In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommercedev1", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

ecommerce stgecommercedev1 ecomm-raw-data


In [0]:
# readChangeFeed flag is used to read the change feed (_change_type column mainly)
df = spark.readStream \
.format("delta") \
.option("readChangeFeed", "true") \
.table(f"{catalog_name}.silver.slv_order_shipments")

#df.limit(10).display()

In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

In [0]:
df_union = df_union.withColumn("carrier_group", F.when((F.col("carrier") == "ECOMEXPRESS") | (F.col("carrier") == "BLUEDART") |
                            (F.col("carrier") == "DELHIVERY") | (F.col("carrier") == "XPRESSBEES"), "Domestic").otherwise("International"))

df_union = df_union.withColumn("is_weekend_shipment", F.when((F.dayofweek(F.col("order_dt")) == 1) | (F.dayofweek(F.col("order_dt")) == 7), True).otherwise(False))

In [0]:
shipments_gold_df = df_union.select(
    F.col("order_dt"),
    F.col("shipment_id"),
    F.col("order_id"),
    F.col("carrier"),
    F.col("carrier_group"),
    F.col("is_weekend_shipment")
)

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_shipments/"
print(gold_checkpoint_path)

def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_shipments"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.order_id = batch_table.order_id",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

shipments_gold_df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

abfss://ecomm-raw-data@stgecommercedev1.dfs.core.windows.net/checkpoint/gold/fact_order_shipments/


In [0]:
spark.sql(f"SELECT max(order_dt) FROM {catalog_name}.gold.gld_fact_order_shipments").show()

+-------------+
|max(order_dt)|
+-------------+
|   2025-09-30|
+-------------+

